# ML Lab Assignment 13: Mall Customer Segmentation

## Objective
Analyze mall customer data and perform customer segmentation using K-Means clustering algorithm.

## Dataset
The Mall_Customers.csv dataset contains the following features:
- CustomerID: Unique ID for each customer
- Gender: Gender of the customer
- Age: Age of the customer
- Annual Income (k$): Annual income in thousands of dollars
- Spending Score (1-100): Score assigned by the mall based on customer behavior and spending

## Task 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Task 2: Load and Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('Mall_Customers.csv')

# Display first few rows
print("First 5 rows of the dataset:")
print(df.head())

In [ ]:
# Display dataset information
print("\nDataset Information:")
print(df.info())

print("\nDataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
print(df.describe())

## Task 3: Data Visualization and Exploratory Data Analysis

In [ ]:
# Gender distribution
plt.figure(figsize=(8, 6))
df['Gender'].value_counts().plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Gender Distribution', fontsize=16)
plt.xlabel('Gender', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Age distribution
plt.figure(figsize=(10, 6))
plt.hist(df['Age'], bins=20, color='skyblue', edgecolor='black')
plt.title('Age Distribution', fontsize=16)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Annual Income distribution
plt.figure(figsize=(10, 6))
plt.hist(df['Annual Income (k$)'], bins=20, color='lightgreen', edgecolor='black')
plt.title('Annual Income Distribution', fontsize=16)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Spending Score distribution
plt.figure(figsize=(10, 6))
plt.hist(df['Spending Score (1-100)'], bins=20, color='lightcoral', edgecolor='black')
plt.title('Spending Score Distribution', fontsize=16)
plt.xlabel('Spending Score (1-100)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Annual Income vs Spending Score
plt.figure(figsize=(10, 6))
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], 
            c='blue', alpha=0.6, s=50)
plt.title('Annual Income vs Spending Score', fontsize=16)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1)
plt.title('Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

## Task 4: Determine Optimal Number of Clusters using Elbow Method

In [ ]:
# Select features for clustering
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Calculate WCSS (Within-Cluster Sum of Squares) for different number of clusters
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--', color='blue', linewidth=2, markersize=8)
plt.title('Elbow Method for Optimal K', fontsize=16)
plt.xlabel('Number of Clusters', fontsize=12)
plt.ylabel('WCSS', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("WCSS values for different K:")
for i, value in enumerate(wcss, 1):
    print(f"K={i}: WCSS={value:.2f}")

## Task 5: Apply K-Means Clustering

In [ ]:
# Apply K-Means with optimal number of clusters (typically 5 for this dataset)
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X)

# Add cluster labels to the dataframe
df['Cluster'] = y_kmeans

print(f"\nClustering completed with {optimal_k} clusters")
print(f"\nCluster distribution:")
print(df['Cluster'].value_counts().sort_index())

## Task 6: Visualize the Clusters

In [ ]:
# Visualize the clusters
plt.figure(figsize=(12, 8))

# Define colors for clusters
colors = ['red', 'blue', 'green', 'cyan', 'magenta']

# Plot each cluster
for i in range(optimal_k):
    plt.scatter(X[y_kmeans == i, 0], X[y_kmeans == i, 1], 
                s=100, c=colors[i], alpha=0.6, 
                label=f'Cluster {i+1}')

# Plot centroids
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            s=300, c='yellow', marker='*', edgecolor='black', linewidth=2,
            label='Centroids')

plt.title('Customer Segments', fontsize=16, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Task 7: Analyze Each Cluster

In [ ]:
# Analyze cluster characteristics
print("\n=== Cluster Analysis ===")
print("\nCluster Statistics:")

cluster_analysis = df.groupby('Cluster').agg({
    'Age': ['mean', 'min', 'max'],
    'Annual Income (k$)': ['mean', 'min', 'max'],
    'Spending Score (1-100)': ['mean', 'min', 'max'],
    'CustomerID': 'count'
}).round(2)

cluster_analysis.columns = ['_'.join(col).strip() for col in cluster_analysis.columns.values]
cluster_analysis.rename(columns={'CustomerID_count': 'Customer_Count'}, inplace=True)

print(cluster_analysis)

In [ ]:
# Visualize cluster characteristics
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Age distribution by cluster
df.boxplot(column='Age', by='Cluster', ax=axes[0, 0])
axes[0, 0].set_title('Age Distribution by Cluster')
axes[0, 0].set_xlabel('Cluster')
axes[0, 0].set_ylabel('Age')

# Annual Income distribution by cluster
df.boxplot(column='Annual Income (k$)', by='Cluster', ax=axes[0, 1])
axes[0, 1].set_title('Annual Income Distribution by Cluster')
axes[0, 1].set_xlabel('Cluster')
axes[0, 1].set_ylabel('Annual Income (k$)')

# Spending Score distribution by cluster
df.boxplot(column='Spending Score (1-100)', by='Cluster', ax=axes[1, 0])
axes[1, 0].set_title('Spending Score Distribution by Cluster')
axes[1, 0].set_xlabel('Cluster')
axes[1, 0].set_ylabel('Spending Score (1-100)')

# Cluster size
cluster_counts = df['Cluster'].value_counts().sort_index()
axes[1, 1].bar(cluster_counts.index, cluster_counts.values, color=colors)
axes[1, 1].set_title('Number of Customers per Cluster')
axes[1, 1].set_xlabel('Cluster')
axes[1, 1].set_ylabel('Number of Customers')

plt.suptitle('')  # Remove the default suptitle
plt.tight_layout()
plt.show()

## Task 8: Customer Segment Interpretation

In [ ]:
# Interpret each cluster
print("\n=== Customer Segment Interpretation ===")
print("\nBased on the clustering analysis, we can identify the following customer segments:\n")

for cluster_id in range(optimal_k):
    cluster_data = df[df['Cluster'] == cluster_id]
    avg_income = cluster_data['Annual Income (k$)'].mean()
    avg_spending = cluster_data['Spending Score (1-100)'].mean()
    count = len(cluster_data)
    
    print(f"Cluster {cluster_id}:")
    print(f"  - Size: {count} customers ({count/len(df)*100:.1f}%)")
    print(f"  - Average Income: ${avg_income:.2f}k")
    print(f"  - Average Spending Score: {avg_spending:.2f}")
    
    # Segment interpretation logic
    if avg_income < 50 and avg_spending < 50:
        segment_type = "Sensible Customers (Low Income, Low Spending)"
    elif avg_income < 50 and avg_spending >= 50:
        segment_type = "Careless Customers (Low Income, High Spending)"
    elif avg_income >= 50 and avg_spending < 50:
        segment_type = "Careful Customers (High Income, Low Spending)"
    elif avg_income >= 50 and avg_spending >= 50:
        segment_type = "Target Customers (High Income, High Spending)"
    else:
        segment_type = "Standard Customers (Average Income & Spending)"
    
    print(f"  - Segment Type: {segment_type}")
    print()

## Task 9: 3D Visualization (Optional)

In [ ]:
# 3D visualization with Age, Income, and Spending Score
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

for i in range(optimal_k):
    cluster_data = df[df['Cluster'] == i]
    ax.scatter(cluster_data['Age'], 
               cluster_data['Annual Income (k$)'], 
               cluster_data['Spending Score (1-100)'],
               c=colors[i], s=60, alpha=0.6, label=f'Cluster {i}')

ax.set_xlabel('Age', fontsize=10)
ax.set_ylabel('Annual Income (k$)', fontsize=10)
ax.set_zlabel('Spending Score (1-100)', fontsize=10)
ax.set_title('3D Customer Segmentation', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## Task 10: Conclusions and Business Recommendations

In [ ]:
print("\n=== Conclusions and Business Recommendations ===")
print("\n1. Customer Segmentation Summary:")
print(f"   - Successfully segmented {len(df)} customers into {optimal_k} distinct groups")
print("   - Each segment shows unique characteristics in income and spending patterns")

print("\n2. Key Insights:")
print("   - Target Customers: Focus marketing efforts on high-income, high-spending customers")
print("   - Careful Customers: Develop strategies to increase spending among high-income, low-spending customers")
print("   - Careless Customers: Provide budget-friendly options for low-income, high-spending customers")
print("   - Sensible Customers: Offer value-for-money products and promotions")

print("\n3. Business Recommendations:")
print("   - Personalize marketing campaigns based on cluster characteristics")
print("   - Develop loyalty programs targeting high-value customers")
print("   - Create special offers to convert low-spending customers to high-spending ones")
print("   - Optimize product placement and inventory based on segment preferences")

print("\n4. Model Performance:")
print(f"   - Final WCSS: {wcss[optimal_k-1]:.2f}")
print(f"   - Number of iterations: {kmeans.n_iter_}")
print(f"   - Algorithm converged successfully")

## Task 11: Save Results

In [ ]:
# Save the clustered data to a new CSV file
df.to_csv('Mall_Customers_Segmented.csv', index=False)
print("\nSegmented customer data saved to 'Mall_Customers_Segmented.csv'")

# Display final dataframe with clusters
print("\nFinal dataset with cluster assignments:")
print(df.head(10))